In [0]:
import dataiku

client = dataiku.api_client()

# Set this to the top-level folder you want to investigate
PARENT_FOLDER_PATH = "/Sandbox" 

print(f"--- Auditing Path: {PARENT_FOLDER_PATH} ---\n")

# 1. Audit SUB-FOLDERS inside the parent
folders = client.list_project_folders()

def audit_folder_recursive(folder_list, target_path):
    for folder in folder_list:
        # Check if this folder is inside our target path
        if folder['path'].startswith(target_path):
            print(f"[FOLDER] Path: {folder['path']}")
            
            # Get permissions for this specific folder
            # Note: We use the raw folder data from the list
            perms = folder.get('permissions', [])
            groups = [p['group'] for p in perms]
            print(f"  - Groups with access: {', '.join(groups) if groups else 'NONE (Inherited)'}")
            print("-" * 30)
            
        # Recurse into children
        if 'children' in folder:
            audit_folder_recursive(folder['children'], target_path)

audit_folder_recursive(folders, PARENT_FOLDER_PATH)

# 2. Audit PROJECTS inside the target path
projects = client.list_projects()
for p_summary in projects:
    project = client.get_project(p_summary['projectKey'])
    metadata = project.get_metadata()
    folder_path = metadata.get('folder', '/')
    
    if folder_path.startswith(PARENT_FOLDER_PATH):
        perms = project.get_permissions()
        groups = [p['group'] for p in perms['permissions']]
        
        print(f"[PROJECT] Name: {p_summary['name']} ({p_summary['projectKey']})")
        print(f"  - Location: {folder_path}")
        print(f"  - Groups with access: {', '.join(groups)}")
        
        # Check for the 'Discoverable' trap
        if perms.get('readerGroup'):
             print(f"  - WARNING: DISCOVERABLE BY: {perms['readerGroup']}")
        print("-" * 30)

In [0]:
import requests
from datetime import datetime

def get_support_escalations():
    ids, API_KEY = ["P2OE4T3", "PWVQZNH"], "u+xqRscqCG91yXoHxL3g"
    
    params = {
        "escalation_policy_ids[]": ids,
        "statuses[]": ["triggered", "acknowledged", "resolved"],
        "limit": 15,
        "sort_by": "created_at:desc"
    }
    
    res = requests.get("https://api.pagerduty.com/incidents", 
                       headers={"Authorization": f"Token token={API_KEY}", 
                                "Accept": "application/vnd.pagerduty+json;version=2"}, 
                       params=params)

    incidents = res.json().get('incidents', [])
    
    #Force Sort (start), remove as needed 
    incidents.sort(key=lambda x: x['created_at'], reverse=True)
    #Force Sort (end)
    
    print(f"{'#'*30}\n{'LATEST ESCALATIONS':^30}\n{'#'*30}\n")

    for i, inc in enumerate(incidents, 1):
        dt = datetime.strptime(inc['created_at'], "%Y-%m-%dT%H:%M:%SZ").strftime("%b %d, %H:%M")
        print(f"{i}. [{inc['status'].upper()}] #{inc['incident_number']} | {dt}")
        print(f"   Summary: {inc['title'][:70]}...")
        print(f"   Link:    {inc['html_url']}\n" + "-"*50)

get_support_escalations()

In [0]:
#test script for support_low_incidents P2OE4T3
import dataiku
import pandas as pd
import requests
from datetime import datetime

# init
API_KEY = "u+xqRscqCG91yXoHxL3g"
POLICY_ID = "P2OE4T3"
headers = {
    "Accept": "application/vnd.pagerduty+json;version=2",
    "Authorization": f"Token token={API_KEY}"
}

# params
params = {
    "escalation_policy_ids[]": [POLICY_ID],
    "statuses[]": ["triggered", "acknowledged", "resolved"],
    "limit": 50,
    "sort_by": "created_at:desc"
}

# get call
res = requests.get("https://api.pagerduty.com/incidents", headers=headers, params=params)
incidents = res.json().get('incidents', [])

# table format conversion
rows = []
for inc in incidents:
    rows.append({
        "incident_number": inc.get('incident_number'),
        "status": inc.get('status'),
        "created_at": inc.get('created_at'),
        "title": inc.get('title'),
        "url": inc.get('html_url')
    })

# write 
df = pd.DataFrame(rows)
output_dataset = dataiku.Dataset("support_low_incidents")
output_dataset.write_with_schema(df)

In [0]:
#test script for support_high_incidents PWVQZNH
import dataiku
import pandas as pd
import requests

# init
API_KEY = "u+xqRscqCG91yXoHxL3g"
POLICY_ID = "PWVQZNH"
headers = {
    "Accept": "application/vnd.pagerduty+json;version=2",
    "Authorization": f"Token token={API_KEY}"
}

# params
params = {
    "escalation_policy_ids[]": [POLICY_ID],
    "statuses[]": ["triggered", "acknowledged", "resolved"],
    "limit": 50,
    "sort_by": "created_at:desc"
}

# get call
res = requests.get("https://api.pagerduty.com/incidents", headers=headers, params=params)
incidents = res.json().get('incidents', [])

# table format conversion
rows = []
for inc in incidents:
    rows.append({
        "incident_number": inc.get('incident_number'),
        "status": inc.get('status'),
        "created_at": inc.get('created_at'),
        "title": inc.get('title'),
        "url": inc.get('html_url')
    })

# write 
df = pd.DataFrame(rows)
output_dataset = dataiku.Dataset("support_high_incidents")
output_dataset.write_with_schema(df)